# Migrate to Apache Iceberg
## In-place Migration of Apache Hudi Tables Iceberg add_files procedure

### Configure Spark session with proper Hudi and Iceberg settings

In [ ]:
%%configure -f
{   
    "conf":{
        "numExecutors": 1, 
        "executorMemory": "14G",
        "executorCores": 4, 
        "spark.dynamicAllocation.enabled": "false",
        "spark.sql.extensions": "org.apache.spark.sql.hudi.HoodieSparkSessionExtension,org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
        "spark.sql.catalog.iceberg_catalog": "org.apache.iceberg.spark.SparkCatalog",
        "spark.sql.catalog.iceberg_catalog.type": "glue",
        "spark.hadoop.hive.metastore.client.factory.class": "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory",
        "spark.jars": "/usr/lib/hudi/hudi-spark-bundle.jar,/usr/lib/hudi/hudi-aws-bundle.jar,/usr/lib/hudi/hudi-utilities-bundle.jar",
        "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
        "spark.sql.catalog.spark_catalog":"org.apache.spark.sql.hudi.catalog.HoodieCatalog",
        "spark.kryo.registrator":"org.apache.spark.HoodieSparkKryoRegistrar"
    }
}

### Configure variables

In [ ]:
hudi_table_name_cow = "hudi_table_test_add_cow"
iceberg_table_name_cow="hudi_table_test_add_cow_iceberg"
hudi_table_name_mor = "hudi_table_test_add_mor"
iceberg_table_name_mor="hudi_table_test_add_mor_iceberg"
bucket_name="mybucket"
db_name="mydb"
##dynamodb table that will be created to handle compaction locks
ddb_table_name="hudi_mor_locks"
region="eu-west-1"

### 1. Hudi COW table
#### Hudi table creation and population

In [ ]:
spark.sql(f"""
CREATE TABLE {db_name}.{hudi_table_name_cow} (
  id INT,
  country STRING,
  name STRING
)
USING hudi
PARTITIONED BY (country)
OPTIONS (
  type = 'cow',
  primaryKey = 'id',
  preCombineField = 'id'
)
LOCATION 's3://{bucket_name}/{db_name}/{hudi_table_name_cow}/'
""")

In [ ]:
spark.sql(f"""
INSERT INTO {db_name}.{hudi_table_name_cow} VALUES 
  (1, 'John Smith', 'USA'),
  (2, 'Mary Johnson', 'USA'),
  (3, 'Marco Rossi', 'Italy'),
  (4, 'Sofia Bianchi', 'Italy');
""")

In [ ]:
spark.sql(f"""
UPDATE {db_name}.{hudi_table_name_cow}
SET name = 
  CASE 
    WHEN id = 1 THEN 'John2'
    WHEN id = 4 THEN 'Sofia2'
    ELSE name
  END
WHERE id IN (1, 4)
""")

#### Migration
1. Create an Iceberg table with the same schema

In [ ]:
spark.sql(f"""
CREATE TABLE iceberg_catalog.{db_name}.{iceberg_table_name_cow} (
  id INT,
  country STRING,
  name STRING
)
USING iceberg
PARTITIONED BY (country)
LOCATION 's3://{bucket_name}/{db_name}/{iceberg_table_name_cow}/'
""")

2. Run Hudi CLEAN operation to keep only the latest files versions for each file group

In [ ]:
spark.sql(f"""
CALL spark_catalog.system.run_clean(
  table => '{db_name}.{hudi_table_name_cow}',
  clean_policy => 'KEEP_LATEST_FILE_VERSIONS',
  file_versions_retained => 1
)
""").show(truncate=False)

In [ ]:
spark.sql(f"""
CALL spark_catalog.system.run_clean(
  table => '{db_name}.{hudi_table_name_cow}',
  clean_policy => 'KEEP_LATEST_FILE_VERSIONS',
  file_versions_retained => 1
)
""").show(truncate=False)

3. Use the add_files procedure to do in-place migration

In [ ]:
spark.sql(f""" 
CALL iceberg_catalog.system.add_files( 
source_table => '`parquet`.`s3://{bucket_name}/{db_name}/{hudi_table_name_cow}/`',  
table => 'iceberg_catalog.{db_name}.{iceberg_table_name_cow}'
)
""").show(truncate=False)

4. Confirm the 2 tables are identical

In [ ]:
spark.sql(f"SELECT id,country,name FROM {db_name}.{hudi_table_name_cow} ORDER BY id").show(truncate=False)

In [ ]:
spark.sql(f"SELECT * FROM iceberg_catalog.{db_name}.{iceberg_table_name_cow} ORDER BY id").show(truncate=False)

### 2. Hudi MOR table
#### Hudi table creation and population

In [ ]:
spark.sql(f"""
CREATE TABLE {db_name}.{hudi_table_name_mor} (
  id INT,
  country STRING,
  name STRING
)
USING hudi
PARTITIONED BY (country)
OPTIONS (
  type = 'mor',
  primaryKey = 'id',
  preCombineField = 'id'
)
LOCATION 's3://{bucket_name}/{db_name}/{hudi_table_name_mor}/'
""")

In [ ]:
spark.sql(f"""
INSERT INTO {db_name}.{hudi_table_name_mor} VALUES 
  (1, 'John Smith', 'USA'),
  (2, 'Mary Johnson', 'USA'),
  (3, 'Marco Rossi', 'Italy'),
  (4, 'Sofia Bianchi', 'Italy');
""")

In [ ]:
spark.sql(f"""
UPDATE {db_name}.{hudi_table_name_mor}
SET name = 
  CASE 
    WHEN id = 1 THEN 'John2'
    WHEN id = 4 THEN 'Sofia2'
    ELSE name
  END
WHERE id IN (1, 4)
""")

#### Migration
1. Create an Iceberg table with the same schema

In [ ]:
spark.sql(f"""
CREATE TABLE iceberg_catalog.{db_name}.{iceberg_table_name_mor} (
  id INT,
  country STRING,
  name STRING
)
USING iceberg
PARTITIONED BY (country)
LOCATION 's3://{bucket_name}/{db_name}/{iceberg_table_name_mor}/'
""")

2. Compact all Hudi table file groups

In [ ]:
#Schedule the compaction 

spark.sql(f""" 
  CALL spark_catalog.system.run_compaction( 
    table => '{db_name}.{hudi_table_name_mor}', 
    op => 'schedule', 
    options =>  
     'hoodie.compact.inline.max.delta.commits=1, 
      hoodie.compact.inline.max.delta.seconds=0, 
      hoodie.write.lock.provider=org.apache.hudi.aws.transaction.lock.DynamoDBBasedLockProvider, 
      hoodie.write.lock.dynamodb.table={ddb_table_name}, 
      hoodie.write.lock.dynamodb.region={region}' 
  ) 
""").show(truncate=False)

In [ ]:
#Execute the compaction 

spark.sql(f""" 
  CALL spark_catalog.system.run_compaction( 
    table => '{db_name}.{hudi_table_name_mor}', 
    op => 'run', 
    options =>  
     'hoodie.write.lock.provider=org.apache.hudi.aws.transaction.lock.DynamoDBBasedLockProvider, 
      hoodie.write.lock.dynamodb.table={ddb_table_name}, 
      hoodie.write.lock.dynamodb.region={region}' 
  ) 
""").show(truncate=False)

3. Run Hudi CLEAN operation to keep only the latest files versions for each file group

In [ ]:
spark.sql(f"""
CALL spark_catalog.system.run_clean(
  table => '{db_name}.{hudi_table_name_mor}',
  clean_policy => 'KEEP_LATEST_FILE_VERSIONS',
  file_versions_retained => 1
)
""").show(truncate=False)

4. Use the add_files procedure to do in-place migration

In [ ]:
spark.sql(f""" 
CALL iceberg_catalog.system.add_files( 
source_table => '`parquet`.`s3://{bucket_name}/{db_name}/{hudi_table_name_mor}/`',  
table => 'iceberg_catalog.{db_name}.{iceberg_table_name_mor}'
)
""").show(truncate=False)

5. Confirm the 2 tables are identical

In [ ]:
spark.sql(f"SELECT id,country,name FROM {db_name}.{hudi_table_name_mor} ORDER BY id").show(truncate=False)

In [ ]:
spark.sql(f"SELECT * FROM iceberg_catalog.{db_name}.{iceberg_table_name_mor} ORDER BY id").show(truncate=False)